### Week 6, Day 2

Before proceeding with our own MCP Server, let's just look at 2 popular marketplaces for what's out there:

https://glama.ai/mcp  
https://smithery.ai/servers 

We're about to create and use our own MCP Server!

It's pretty simple, but it's not super-simple. The excitement around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

backend/accounts.py

In [6]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from rich import print

load_dotenv(override=True)

True

In [7]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Any guesses where this Account python module came from?!

I didn't write it!

In [8]:
from backend.accounts import Account

In [9]:
account = Account.get("Ed")
account.reset()
print(account)


Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [10]:
print(account.buy_shares("AMZN", 3, "Because this bookstore website looks promising"))

Completed. Latest details:
{"name": "ed", "balance": 9208.27972, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", 
"quantity": 3, "price": 263.90676, "timestamp": "2026-10-01 20:31:25", "rationale": "Because this bookstore website
looks promising"}], "portfolio_value_time_series": [["2026-10-01 20:31:25", 9998.41972]], "total_portfolio_value": 
9998.41972, "total_profit_loss": -1.5802800000001298}

In [12]:
print(account.report())

{"name": "ed", "balance": 9208.27972, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", 
"quantity": 3, "price": 263.90676, "timestamp": "2026-10-01 20:31:25", "rationale": "Because this bookstore website
looks promising"}], "portfolio_value_time_series": [["2026-10-01 20:31:25", 9998.41972], ["2026-10-01 20:31:39", 
9998.35972], ["2026-10-01 20:31:46", 9998.32972]], "total_portfolio_value": 9998.32972, "total_profit_loss": 
-1.6702800000002753}

In [14]:
print(account.list_transactions())

[
    {
        'symbol': 'AMZN',
        'quantity': 3,
        'price': 263.90676,
        'timestamp': '2026-10-01 20:31:25',
        'rationale': 'Because this bookstore website looks promising'
    }
]

### Now we write an MCP server and use it directly!

In [15]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "-m", "backend.accounts_server"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [17]:
print(mcp_tools)

[
    Tool(
        name='get_balance',
        title=None,
        description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the 
account holder\n    ',
        inputSchema={
            'properties': {'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name'],
            'title': 'get_balanceArguments',
            'type': 'object'
        },
        outputSchema={
            'properties': {'result': {'title': 'Result', 'type': 'number'}},
            'required': ['result'],
            'title': 'get_balanceOutput',
            'type': 'object'
        },
        icons=None,
        annotations=None,
        meta=None,
        execution=None
    ),
    Tool(
        name='get_holdings',
        title=None,
        description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the 
account holder\n    ',
        inputSchema={
            'properties': {'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name'],
            'title': 'get_holdingsArguments',
            'type': 'object'
        },
        outputSchema={
            'additionalProperties': {'type': 'integer'},
            'title': 'get_holdingsDictOutput',
            'type': 'object'
        },
        icons=None,
        annotations=None,
        meta=None,
        execution=None
    ),
    Tool(
        name='buy_shares',
        title=None,
        description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        
symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale 
for the purchase and fit with the account's strategy\n    ",
        inputSchema={
            'properties': {
                'name': {'title': 'Name', 'type': 'string'},
                'symbol': {'title': 'Symbol', 'type': 'string'},
                'quantity': {'title': 'Quantity', 'type': 'integer'},
                'rationale': {'title': 'Rationale', 'type': 'string'}
            },
            'required': ['name', 'symbol', 'quantity', 'rationale'],
            'title': 'buy_sharesArguments',
            'type': 'object'
        },
        outputSchema={
            'properties': {'result': {'title': 'Result', 'type': 'number'}},
            'required': ['result'],
            'title': 'buy_sharesOutput',
            'type': 'object'
        },
        icons=None,
        annotations=None,
        meta=None,
        execution=None
    ),
    Tool(
        name='sell_shares',
        title=None,
        description="Sell shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        
symbol: The symbol of the stock\n        quantity: The quantity of shares to sell\n        rationale: The rationale
for the sale and fit with the account's strategy\n    ",
        inputSchema={
            'properties': {
                'name': {'title': 'Name', 'type': 'string'},
                'symbol': {'title': 'Symbol', 'type': 'string'},
                'quantity': {'title': 'Quantity', 'type': 'integer'},
                'rationale': {'title': 'Rationale', 'type': 'string'}
            },
            'required': ['name', 'symbol', 'quantity', 'rationale'],
            'title': 'sell_sharesArguments',
            'type': 'object'
        },
        outputSchema={
            'properties': {'result': {'title': 'Result', 'type': 'number'}},
            'required': ['result'],
            'title': 'sell_sharesOutput',
            'type': 'object'
        },
        icons=None,
        annotations=None,
        meta=None,
        execution=None
    ),
    Tool(
        name='change_strategy',
        title=None,
        description='At your discretion, if you choose to, call this to change your investment strategy for the 
future.\n\n    Args:\n        name: The name of the account holder\n        strategy: The new strategy for the 
account\n    ',
        inputSchem

In [18]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-5.4-mini"

In [19]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Your account balance is **$9,208.28**.

Your holdings are:
- **AMZN**: 3 shares

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to send a push notification, and then
            enjoy the outcome! There is a solution in backend/push_server.py if you need a hint.
            </span>
        </td>
    </tr>
</table>